## This file is used for applying cleaning + transformations on categorical data before finalizing the code for pipeline.

###   1. emp_length  

In [38]:
import logging
import numpy as np
import pandas as pd

logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)

if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(
        logging.Formatter("%(asctime)s | %(levelname)s | %(name)s | %(message)s",
                          datefmt="%Y-%m-%d %H:%M:%S")
    )
    logger.addHandler(handler)

In [40]:
file_path = '../data/processed/loan_selected.parquet'

def read_data(file_path = file_path):
    """Reads the dataset from the preprocessed data - Parquet file."""
    logger.info(f"Reading data from {file_path}...")
    try: 
        df = pd.read_parquet(file_path)
        logger.info("Successfully read the file")
        return df

    except Exception as e:
        logger.error(f"Error reading data: {e}")
        raise e 



df = read_data()
df_copy = df.copy()
df.sample(5)


2026-03-11 02:09:33 | INFO | __main__ | Reading data from ../data/processed/loan_selected.parquet...
2026-03-11 02:09:33 | INFO | __main__ | Successfully read the file


,loan_amnt,term,emp_length,home_ownership,annual_inc,verification_status,target,purpose,addr_state,dti,...,mo_sin_old_rev_tl_op,mort_acc,num_actv_rev_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,percent_bc_gt_75,pub_rec_bankruptcies,tot_hi_cred_lim,total_bc_limit
289249,6400,36 months,5 years,MORTGAGE,20000.0,Verified,0,debt_consolidation,TN,28.51,...,94.0,0.0,3.0,3.0,7.0,3.0,100.0,0.0,24360.0,700.0
1296635,16000,36 months,< 1 year,RENT,103000.0,Not Verified,0,credit_card,IN,11.80,...,112.0,0.0,4.0,6.0,8.0,4.0,20.0,0.0,92646.0,61300.0
1204457,15000,36 months,10+ years,MORTGAGE,58000.0,Verified,0,credit_card,MN,26.53,...,154.0,3.0,6.0,7.0,10.0,6.0,75.0,0.0,158841.0,11100.0
1246343,10000,36 months,8 years,MORTGAGE,66000.0,Source Verified,0,home_improvement,CA,9.60,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN
531369,19000,36 months,10+ years,MORTGAGE,75000.0,Not Verified,0,debt_consolidation,PA,28.69,...,159.0,5.0,3.0,6.0,17.0,3.0,0.0,0.0,356012.0,57300.0


In [34]:

# --- emp_length ---
EMP_LENGTH_MAP = {
    "< 1 year":  0,
    "1 year":    1,
    "2 years":   2,
    "3 years":   3,
    "4 years":   4,
    "5 years":   5,
    "6 years":   6,
    "7 years":   7,
    "8 years":   8,
    "9 years":   9,
    "10+ years": 10,
}

EMP_LENGTH_BINS   = [0, 2, 5, 10]
EMP_LENGTH_LABELS = ["unstable", "transitional", "stable"]

def clean_emp_length(df: pd.DataFrame) -> pd.DataFrame:
    """
    Converts emp_length string to stability bins in 3 steps:

    Step 1 — Map string → integer (0–10)
              Unknown strings → NaN (median imputer handles downstream)
    Step 2 — clip(upper=10) so any value > 10 maps to "stable"
              (bank/customer may enter 12, 15, 20 years — all mean 10+)
    Step 3 — pd.cut into 3 bins:
              [0–2]  → "unstable"
              [3–5]  → "transitional"
              [6–10] → "stable"

    New column : emp_length_stability (ordinal string)
    Dropped    : emp_length, emp_length_num  (at end of master caller)
    """
    logger.info("Cleaning 'emp_length' → 'emp_length_stability'")

    # Step 1 — string → integer
    df["emp_length_num"] = df["emp_length"].map(EMP_LENGTH_MAP)
    unmapped = df["emp_length_num"].isna().sum()
    if unmapped > 0:
        logger.warning(f"  {unmapped} rows unmapped or originally NaN → left as NaN for median imputer")

    # STEP 2 - Median Impute 
    logger.info(f"Median Imputing {unmapped} rows ")
    median_val = df["emp_length_num"].median()
    df["emp_length_num"] = df["emp_length_num"].fillna(median_val)

    # Step 3 — clip: anything above 10 → 10 so it falls in "stable" bin
    df["emp_length_num"] = df["emp_length_num"].clip(upper=10)
    logger.debug(f"  After clip — min={df['emp_length_num'].min()}, max={df['emp_length_num'].max()}")

    # Step 4 — bin into stability categories
    df["emp_length_stability"] = pd.cut(
        df["emp_length_num"],
        bins=EMP_LENGTH_BINS,
        labels=EMP_LENGTH_LABELS,
        include_lowest=True,  # ensures 0 → "unstable" (left edge included)
        right=True,           # bins are (left, right] — 10 included in stable
    )

    # Drop original string col + intermediate integer col — no longer needed
    df.drop(columns=["emp_length", "emp_length_num"], inplace=True)
    logger.info("  Dropped: 'emp_length', 'emp_length_num'")

   
    return df

clean_emp_length(df_copy)
df_copy.emp_length_stability

2026-03-11 01:00:52 | INFO | __main__ | Cleaning 'emp_length' → 'emp_length_stability'
2026-03-11 01:00:52 | WARNING | __main__ |   75457 rows unmapped or originally NaN → left as NaN for median imputer
2026-03-11 01:00:52 | INFO | __main__ | Median Imputing 75457 rows 
2026-03-11 01:00:52 | DEBUG | __main__ |   After clip — min=0.0, max=10.0
2026-03-11 01:00:52 | INFO | __main__ |   Dropped: 'emp_length', 'emp_length_num'


0          transitional
1              unstable
2                stable
3                stable
4          transitional
               ...     
1303633        unstable
1303634          stable
1303635          stable
1303636        unstable
1303637        unstable
Name: emp_length_stability, Length: 1303638, dtype: category
Categories (3, object): ['unstable' < 'transitional' < 'stable']

### 2. Home Ownership 

In [41]:

HOME_OWNERSHIP_MERGE = {"ANY": "OWN", "OTHER": "OWN"}
HOME_OWNERSHIP_VALID = ["RENT", "OWN", "MORTGAGE"]

def clean_home_ownership(df: pd.DataFrame) -> pd.DataFrame:
    """
    Cleans home_ownership to 3 valid categories in-place.

    Step 1 — Merge ANY → OWN, OTHER → OWN (similar default rates ~19%)
    Step 2 — Result: RENT / OWN / MORTGAGE only

    Modified in-place : home_ownership (same column name, clean values)
    No new columns    : encoding handled in encoder.py
    """
    logger.info("Cleaning 'home_ownership' → RENT / OWN / MORTGAGE")

    # Step 1 — merge ANY + OTHER → OWN
    df["home_ownership"] = df["home_ownership"].replace(HOME_OWNERSHIP_MERGE)
    logger.info(f"  Merged ANY → OWN, OTHER → OWN")

    # Step 2 - Verify only valid categories remain
    unexpected = set(df["home_ownership"].unique()) - set(HOME_OWNERSHIP_VALID)
    if unexpected:
        logger.warning(f"  Unexpected categories still present: {unexpected}")

    logger.info(f"  Distribution: {df['home_ownership'].value_counts().to_dict()}")
    return df

df_copy = clean_home_ownership(df_copy)
df_copy.home_ownership.value_counts()



2026-03-11 02:09:44 | INFO | __main__ | Cleaning 'home_ownership' → RENT / OWN / MORTGAGE
2026-03-11 02:09:44 | INFO | __main__ |   Merged ANY → OWN, OTHER → OWN
2026-03-11 02:09:44 | INFO | __main__ |   Distribution: {'MORTGAGE': 645509, 'RENT': 517821, 'OWN': 140260}


home_ownership
MORTGAGE    645509
RENT        517821
OWN         140260
Name: count, dtype: int64

### 3. Purpose 

In [42]:
# --- purpose risk buckets ---
PURPOSE_BUCKET_MAP = {
    # High risk — default rate > 22%
    "small_business":    "high_risk",
    "renewable_energy":  "high_risk",
    "moving":            "high_risk",
    # Medium risk — default rate 18–22%
    "medical":           "medium_risk",
    "house":             "medium_risk",
    "debt_consolidation":"medium_risk",
    "other":             "medium_risk",
    "vacation":          "medium_risk",
    "major_purchase":    "medium_risk",
    # Low risk — default rate < 18%
    "home_improvement":  "low_risk",
    "educational":       "low_risk",
    "credit_card":       "low_risk",
    "car":               "low_risk",
    "wedding":           "low_risk",
}

def clean_purpose(df: pd.DataFrame) -> pd.DataFrame:
    """
    Groups 14 purpose categories into 3 risk buckets based on default rates:
      high_risk   → > 22%  : small_business, renewable_energy, moving
      medium_risk → 18-22% : medical, house, debt_consolidation, other,
                             vacation, major_purchase
      low_risk    → < 18%  : home_improvement, educational, credit_card,
                             car, wedding

    Unknown categories at inference → "medium_risk" (safe default)

    Modified in-place : purpose column replaced by purpose_bucket
    Dropped           : original purpose column
    """
    logger.info("Cleaning 'purpose' → 'purpose_bucket' (3 risk buckets)")

    df["purpose_bucket"] = df["purpose"].map(PURPOSE_BUCKET_MAP)

    # Handle unseen categories at inference time → default to medium_risk
    unseen = df["purpose_bucket"].isna().sum()
    if unseen > 0:
        logger.warning(f"  {unseen} rows had unknown purpose values → defaulting to 'medium_risk'")
        df["purpose_bucket"] = df["purpose_bucket"].fillna("medium_risk")

    logger.info(f"  Distribution: {df['purpose_bucket'].value_counts().to_dict()}")

    df.drop(columns=["purpose"], inplace=True)
    logger.info("  Dropped: 'purpose'")

    return df

df_copy  = clean_purpose(df_copy)
df_copy.purpose_bucket.value_counts()


2026-03-11 02:23:22 | INFO | __main__ | Cleaning 'purpose' → 'purpose_bucket' (3 risk buckets)
2026-03-11 02:23:22 | INFO | __main__ |   Distribution: {'medium_risk': 891581, 'low_risk': 386915, 'high_risk': 25094}
2026-03-11 02:23:22 | INFO | __main__ |   Dropped: 'purpose'


purpose_bucket
medium_risk    891581
low_risk       386915
high_risk       25094
Name: count, dtype: int64

### 4. Addr State


In [43]:
# --- addr_state → US regions ---
STATE_REGION_MAP = {
    # Northeast
    "CT": "northeast", "ME": "northeast", "MA": "northeast",
    "NH": "northeast", "RI": "northeast", "VT": "northeast",
    "NJ": "northeast", "NY": "northeast", "PA": "northeast",
    # Southeast
    "DE": "southeast", "FL": "southeast", "GA": "southeast",
    "MD": "southeast", "NC": "southeast", "SC": "southeast",
    "VA": "southeast", "WV": "southeast", "DC": "southeast",
    "AL": "southeast", "KY": "southeast", "MS": "southeast",
    "TN": "southeast", "AR": "southeast", "LA": "southeast",
    "OK": "southeast", "TX": "southeast",
    # Midwest
    "IL": "midwest", "IN": "midwest", "MI": "midwest",
    "OH": "midwest", "WI": "midwest", "IA": "midwest",
    "KS": "midwest", "MN": "midwest", "MO": "midwest",
    "NE": "midwest", "ND": "midwest", "SD": "midwest",
    # West
    "AZ": "west", "CO": "west", "ID": "west", "MT": "west",
    "NV": "west", "NM": "west", "UT": "west", "WY": "west",
    "AK": "west", "CA": "west", "HI": "west", "OR": "west",
    "WA": "west",
}

def clean_addr_state(df: pd.DataFrame) -> pd.DataFrame:
    """
    Maps 51 US state codes to 4 geographic regions:
      northeast / southeast / midwest / west

    Rationale: 51 OHE columns is too high cardinality. Regional grouping
    preserves geographic economic patterns that influence default risk.

    Modified in-place : addr_state replaced by addr_region
    Dropped           : original addr_state column
    """
    logger.info("Cleaning 'addr_state' → 'addr_region' (4 US regions)")

    df["addr_region"] = df["addr_state"].map(STATE_REGION_MAP)

    unmapped = df["addr_region"].isna().sum()
    if unmapped > 0:
        logger.warning(f"  {unmapped} rows had unrecognised state codes → NaN")

    logger.info(f"  Distribution: {df['addr_region'].value_counts().to_dict()}")

    df.drop(columns=["addr_state"], inplace=True)
    logger.info("  Dropped: 'addr_state'")

    return df

df_copy  = clean_addr_state(df_copy)
df_copy.addr_region.value_counts()


2026-03-11 02:34:27 | INFO | __main__ | Cleaning 'addr_state' → 'addr_region' (4 US regions)
2026-03-11 02:34:27 | INFO | __main__ |   Distribution: {'southeast': 463393, 'west': 350277, 'northeast': 262859, 'midwest': 227061}
2026-03-11 02:34:27 | INFO | __main__ |   Dropped: 'addr_state'


addr_region
southeast    463393
west         350277
northeast    262859
midwest      227061
Name: count, dtype: int64

### 5. earliest_cr_line

In [44]:
# --- earliest_cr_line → credit_maturity ---
CREDIT_HISTORY_REFERENCE_YEAR = 2015
CREDIT_MATURITY_BINS   = [0, 5, 15, 30, 65]   # 65 as hard upper after clip
CREDIT_MATURITY_LABELS = ["new", "moderate", "established", "veteran"]

def clean_earliest_cr_line(df: pd.DataFrame) -> pd.DataFrame:
    """
    Converts earliest_cr_line date string into credit_maturity bins only.

    Rationale: credit_history_years/months skipped — highly collinear with
    mo_sin_old_rev_tl_op (already in numerical features). credit_maturity
    captures non-linear threshold effects as ordinal categories instead.

    Step 1 — Parse date string → extract year
    Step 2 — credit_history_years = REFERENCE_YEAR - extracted_year
              clip(lower=0, upper=64) handles:
                - Future dates / bad data → 0 → "new"
                - Very old accounts (1951) → 64 → "veteran"
    Step 3 — pd.cut into 4 maturity bins:
              [0–5)   → "new"
              [5–15)  → "moderate"
              [15–30) → "established"
              [30–65] → "veteran"

    New column : credit_maturity (ordinal string → OHE in encoder.py)
    Dropped    : earliest_cr_line
    """
    logger.info("Cleaning 'earliest_cr_line' → 'credit_maturity'")

    # Step 1 — parse date, extract year
    parsed = pd.to_datetime(df["earliest_cr_line"], format="%b-%Y", errors="coerce")
    nat_count = parsed.isna().sum()
    if nat_count > 0:
        logger.warning(f"  {nat_count} rows could not be parsed → NaT → "
                       f"credit_history_years = NaN → pd.cut → NaN category")

    # Step 2 — compute years, clip to valid range
    credit_history_years = (CREDIT_HISTORY_REFERENCE_YEAR - parsed.dt.year)
    credit_history_years = credit_history_years.clip(lower=0, upper=64)
    logger.debug(f"  credit_history_years — min={credit_history_years.min()}, "
                 f"max={credit_history_years.max()}, "
                 f"nulls={credit_history_years.isna().sum()}")

    # Step 3 — bin into maturity categories
    df["credit_maturity"] = pd.cut(
        credit_history_years,
        bins=CREDIT_MATURITY_BINS,
        labels=CREDIT_MATURITY_LABELS,
        include_lowest=True,  # ensures 0 → "new"
        right=False,          # [left, right) — left edge included
    )

    logger.info(f"  Distribution: {df['credit_maturity'].value_counts().to_dict()}")

    df.drop(columns=["earliest_cr_line"], inplace=True)
    logger.info("  Dropped: 'earliest_cr_line'")

    return df

df_copy  = clean_earliest_cr_line(df_copy)
df_copy.credit_maturity.value_counts()


2026-03-11 02:45:32 | INFO | __main__ | Cleaning 'earliest_cr_line' → 'credit_maturity'
2026-03-11 02:45:32 | DEBUG | __main__ |   credit_history_years — min=0, max=64, nulls=0
2026-03-11 02:45:32 | INFO | __main__ |   Distribution: {'established': 611332, 'moderate': 576157, 'veteran': 82262, 'new': 33839}
2026-03-11 02:45:33 | INFO | __main__ |   Dropped: 'earliest_cr_line'


credit_maturity
established    611332
moderate       576157
veteran         82262
new             33839
Name: count, dtype: int64